# Trabajo 3: Análisis de datos con NumPy y Pandas

## RetailNow — Análisis de ventas, inventarios y satisfacción del cliente

Este notebook procesa los archivos `sales.csv`, `inventories.csv` y `satisfaction.csv` con **Pandas** y realiza cálculos estadísticos y simulaciones con **NumPy**.

Los análisis incluidos cumplen con los puntos solicitados en el ejercicio:

- carga y limpieza de los datos;
- ventas totales por producto y por tienda;
- ingresos totales por tienda;
- resumen estadístico;
- rotación de inventarios e inventarios críticos;
- satisfacción del cliente y relación con las ventas;
- mediana y desviación estándar con NumPy;
- simulación reproducible de ventas futuras.

## 1. Importar las librerías necesarias

In [2]:
!pip install numpy pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl (12.6 MB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Cargar los archivos CSV

La plataforma de evaluación indica las rutas absolutas `/workspace/...`.
Para poder ejecutar el mismo notebook localmente en Visual Studio Code, si esas rutas no existen se utilizará la carpeta `data/` ubicada junto al notebook.

In [5]:
# Rutas requeridas por la plataforma.
ruta_sales = Path("/workspace/sales.csv")
ruta_inventories = Path("/workspace/inventories.csv")
ruta_satisfaction = Path("/workspace/satisfaction.csv")

# Rutas locales del proyecto para trabajar en Visual Studio Code.
if not ruta_sales.exists():
    ruta_sales = Path("data/sales.csv")
    ruta_inventories = Path("data/inventories.csv")
    ruta_satisfaction = Path("data/satisfaction.csv")

ventas = pd.read_csv(ruta_sales)
inventarios = pd.read_csv(ruta_inventories)
satisfaccion = pd.read_csv(ruta_satisfaction)

print("DataFrame de ventas:")
display(ventas.head())

print("\nDataFrame de inventarios:")
display(inventarios.head())

print("\nDataFrame de satisfacción:")
display(satisfaccion.head())

DataFrame de ventas:


,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta
0,1,Producto A,20,100,2023-01-05
1,1,Producto B,15,200,2023-01-06
2,2,Producto A,30,100,2023-01-07
3,2,Producto C,25,300,2023-01-08
4,3,Producto A,10,100,2023-01-09



DataFrame de inventarios:


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización
0,1,Producto A,50,2023-01-05
1,1,Producto B,40,2023-01-06
2,2,Producto A,60,2023-01-07
3,2,Producto C,45,2023-01-08
4,3,Producto A,30,2023-01-09



DataFrame de satisfacción:


,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
0,1,85,2023-01-15
1,2,90,2023-01-15
2,3,70,2023-01-15
3,4,65,2023-01-15
4,5,55,2023-01-15


## 3. Verificar la estructura y limpiar los datos

Los archivos reales proporcionados utilizan estas columnas:

- `sales.csv`: `ID_Tienda`, `Producto`, `Cantidad_Vendida`, `Precio_Unitario`, `Fecha_Venta`
- `inventories.csv`: `ID_Tienda`, `Producto`, `Stock_Disponible`, `Fecha_Actualización`
- `satisfaction.csv`: `ID_Tienda`, `Satisfacción_Promedio`, `Fecha_Evaluación`

Se convierten las fechas a `datetime` y se eliminan las filas con valores nulos mediante `dropna()`.

In [6]:
print("Columnas de ventas:", ventas.columns.tolist())
print("Columnas de inventarios:", inventarios.columns.tolist())
print("Columnas de satisfacción:", satisfaccion.columns.tolist())

print("\nValores nulos antes de limpiar:")
print("Ventas:", ventas.isna().sum().sum())
print("Inventarios:", inventarios.isna().sum().sum())
print("Satisfacción:", satisfaccion.isna().sum().sum())

Columnas de ventas: ['ID_Tienda', 'Producto', 'Cantidad_Vendida', 'Precio_Unitario', 'Fecha_Venta']
Columnas de inventarios: ['ID_Tienda', 'Producto', 'Stock_Disponible', 'Fecha_Actualización']
Columnas de satisfacción: ['ID_Tienda', 'Satisfacción_Promedio', 'Fecha_Evaluación']

Valores nulos antes de limpiar:
Ventas: 0
Inventarios: 0
Satisfacción: 0


In [7]:
# Convertir las columnas de fecha al tipo datetime.
ventas["Fecha_Venta"] = pd.to_datetime(ventas["Fecha_Venta"])
inventarios["Fecha_Actualización"] = pd.to_datetime(
    inventarios["Fecha_Actualización"]
)
satisfaccion["Fecha_Evaluación"] = pd.to_datetime(
    satisfaccion["Fecha_Evaluación"]
)

# Eliminar filas con valores nulos.
ventas = ventas.dropna().copy()
inventarios = inventarios.dropna().copy()
satisfaccion = satisfaccion.dropna().copy()

print("Filas válidas después de aplicar dropna():")
print("Ventas:", len(ventas))
print("Inventarios:", len(inventarios))
print("Satisfacción:", len(satisfaccion))

Filas válidas después de aplicar dropna():
Ventas: 10
Inventarios: 10
Satisfacción: 5


## 4. Exploración y análisis de ventas con Pandas

Primero se crea la columna `Total_Ventas`, calculada como:

**Cantidad_Vendida × Precio_Unitario**

Después se calculan las ventas totales por producto, las ventas totales por tienda y los ingresos totales por tienda.

In [8]:
# Crear la columna monetaria solicitada para el análisis.
ventas["Total_Ventas"] = (
    ventas["Cantidad_Vendida"] * ventas["Precio_Unitario"]
)

print("Ventas con la columna Total_Ventas:")
display(ventas)

Ventas con la columna Total_Ventas:


,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta,Total_Ventas
0,1,Producto A,20,100,2023-01-05,2000
1,1,Producto B,15,200,2023-01-06,3000
2,2,Producto A,30,100,2023-01-07,3000
3,2,Producto C,25,300,2023-01-08,7500
4,3,Producto A,10,100,2023-01-09,1000
5,3,Producto B,40,200,2023-01-10,8000
6,4,Producto C,35,300,2023-01-11,10500
7,4,Producto A,25,100,2023-01-12,2500
8,5,Producto B,20,200,2023-01-13,4000
9,5,Producto C,30,300,2023-01-14,9000


In [9]:
# Ventas totales por producto, medidas en unidades vendidas.
ventas_por_producto = (
    ventas.groupby("Producto", as_index=False)["Cantidad_Vendida"]
    .sum()
    .rename(columns={"Cantidad_Vendida": "Unidades_Totales_Vendidas"})
    .sort_values("Unidades_Totales_Vendidas", ascending=False)
)

print("Ventas totales por producto:")
display(ventas_por_producto)

Ventas totales por producto:


,Producto,Unidades_Totales_Vendidas
2,Producto C,90
0,Producto A,85
1,Producto B,75


In [10]:
# Ventas totales por tienda, medidas en unidades vendidas.
ventas_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Cantidad_Vendida"]
    .sum()
    .rename(columns={"Cantidad_Vendida": "Unidades_Totales_Vendidas"})
    .sort_values("Unidades_Totales_Vendidas", ascending=False)
)

print("Ventas totales por tienda:")
display(ventas_por_tienda)

Ventas totales por tienda:


,ID_Tienda,Unidades_Totales_Vendidas
3,4,60
1,2,55
2,3,50
4,5,50
0,1,35


In [11]:
# Ingresos totales por tienda.
ingresos_por_tienda = (
    ventas.groupby("ID_Tienda", as_index=False)["Total_Ventas"]
    .sum()
    .rename(columns={"Total_Ventas": "Ingresos_Totales"})
    .sort_values("Ingresos_Totales", ascending=False)
)

print("Ingresos totales por tienda:")
display(ingresos_por_tienda)

Ingresos totales por tienda:


,ID_Tienda,Ingresos_Totales
4,5,13000
3,4,13000
1,2,10500
2,3,9000
0,1,5000


### Resumen estadístico de las ventas

`describe()` permite obtener métricas como cantidad, media, desviación estándar, mínimo, percentiles y máximo.

El archivo `sales.csv` proporcionado **no contiene una columna de categoría de producto**, por lo que el análisis opcional de promedio por tienda y categoría no puede realizarse con estos datos sin inventar información.

In [12]:
print("Resumen estadístico:")
display(
    ventas[
        ["Cantidad_Vendida", "Precio_Unitario", "Total_Ventas"]
    ].describe()
)

print("Mediana de Total_Ventas con Pandas:", ventas["Total_Ventas"].median())

Resumen estadístico:


,Cantidad_Vendida,Precio_Unitario,Total_Ventas
count,10.000000,10.000000,10.000000
mean,25.000000,190.000000,5050.000000
std,9.128709,87.559504,3361.960407
min,10.000000,100.000000,1000.000000
25%,20.000000,100.000000,2625.000000
50%,25.000000,200.000000,3500.000000
75%,30.000000,275.000000,7875.000000
max,40.000000,300.000000,10500.000000


Mediana de Total_Ventas con Pandas: 3500.0


## 5. Análisis de inventarios con Pandas

La rotación de inventario se calcula, para cada combinación de tienda y producto, como:

**Cantidad vendida / Stock disponible**

Según el enunciado, se considera crítico un registro cuya proporción vendida sea inferior al **10 %** del stock disponible.

In [13]:
# Sumar las unidades vendidas por tienda y producto.
ventas_tienda_producto = (
    ventas.groupby(
        ["ID_Tienda", "Producto"],
        as_index=False
    )["Cantidad_Vendida"].sum()
)

# Relacionar ventas e inventarios.
inventarios_analisis = inventarios.merge(
    ventas_tienda_producto,
    on=["ID_Tienda", "Producto"],
    how="left"
)

# Si un producto no tiene ventas registradas, se consideran 0 unidades.
inventarios_analisis["Cantidad_Vendida"] = (
    inventarios_analisis["Cantidad_Vendida"].fillna(0)
)

# Calcular la rotación evitando divisiones entre cero.
inventarios_analisis["Rotacion_Inventario"] = np.where(
    inventarios_analisis["Stock_Disponible"] > 0,
    inventarios_analisis["Cantidad_Vendida"]
    / inventarios_analisis["Stock_Disponible"],
    np.nan
)

# Mostrar también la rotación como porcentaje para facilitar la lectura.
inventarios_analisis["Rotacion_Porcentaje"] = (
    inventarios_analisis["Rotacion_Inventario"] * 100
)

print("Rotación de inventario por tienda y producto:")
display(inventarios_analisis)

Rotación de inventario por tienda y producto:


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario,Rotacion_Porcentaje
0,1,Producto A,50,2023-01-05,20,0.400000,40.000000
1,1,Producto B,40,2023-01-06,15,0.375000,37.500000
2,2,Producto A,60,2023-01-07,30,0.500000,50.000000
3,2,Producto C,45,2023-01-08,25,0.555556,55.555556
4,3,Producto A,30,2023-01-09,10,0.333333,33.333333
5,3,Producto B,80,2023-01-10,40,0.500000,50.000000
6,4,Producto C,70,2023-01-11,35,0.500000,50.000000
7,4,Producto A,50,2023-01-12,25,0.500000,50.000000
8,5,Producto B,40,2023-01-13,20,0.500000,50.000000
9,5,Producto C,60,2023-01-14,30,0.500000,50.000000


In [14]:
# Filtrar registros con rotación inferior al 10 %.
inventarios_criticos = inventarios_analisis[
    inventarios_analisis["Rotacion_Inventario"] < 0.10
].copy()

print("Inventarios críticos (< 10 %):")

if inventarios_criticos.empty:
    print("No se encontraron productos con rotación inferior al 10 %.")
else:
    display(inventarios_criticos)

Inventarios críticos (< 10 %):
No se encontraron productos con rotación inferior al 10 %.


In [15]:
# Calcular la rotación promedio por tienda utilizando groupby().
rotacion_por_tienda = (
    inventarios_analisis.groupby(
        "ID_Tienda",
        as_index=False
    )["Rotacion_Inventario"]
    .mean()
)

rotacion_por_tienda["Rotacion_Porcentaje"] = (
    rotacion_por_tienda["Rotacion_Inventario"] * 100
)

print("Rotación promedio de inventario por tienda:")
display(rotacion_por_tienda)

Rotación promedio de inventario por tienda:


,ID_Tienda,Rotacion_Inventario,Rotacion_Porcentaje
0,1,0.387500,38.750000
1,2,0.527778,52.777778
2,3,0.416667,41.666667
3,4,0.500000,50.000000
4,5,0.500000,50.000000


## 6. Satisfacción del cliente

Se analiza la satisfacción promedio de cada tienda y se relaciona con los ingresos totales obtenidos.

El archivo entregado expresa la satisfacción en una escala de **0 a 100**, por lo que se filtran las tiendas con una satisfacción inferior a **60**.

In [16]:
# El archivo ya contiene una satisfacción promedio por tienda.
satisfaccion_por_tienda = satisfaccion[
    ["ID_Tienda", "Satisfacción_Promedio"]
].copy()

print("Satisfacción por tienda:")
display(satisfaccion_por_tienda)

Satisfacción por tienda:


,ID_Tienda,Satisfacción_Promedio
0,1,85
1,2,90
2,3,70
3,4,65
4,5,55


In [17]:
# Relacionar satisfacción e ingresos.
rendimiento_tiendas = ingresos_por_tienda.merge(
    satisfaccion_por_tienda,
    on="ID_Tienda",
    how="inner"
)

print("Relación entre ingresos y satisfacción:")
display(rendimiento_tiendas)

Relación entre ingresos y satisfacción:


,ID_Tienda,Ingresos_Totales,Satisfacción_Promedio
0,5,13000,55
1,4,13000,65
2,2,10500,90
3,3,9000,70
4,1,5000,85


In [18]:
# Filtrar tiendas con satisfacción menor al 60 %.
tiendas_baja_satisfaccion = rendimiento_tiendas[
    rendimiento_tiendas["Satisfacción_Promedio"] < 60
].copy()

print("Tiendas con satisfacción menor al 60 %:")
display(tiendas_baja_satisfaccion)

Tiendas con satisfacción menor al 60 %:


,ID_Tienda,Ingresos_Totales,Satisfacción_Promedio
0,5,13000,55


### Recomendaciones

Para las tiendas con satisfacción inferior al 60 %, se recomienda revisar la atención al cliente, la disponibilidad de productos y los tiempos de servicio. También conviene analizar comentarios de clientes y comparar su rendimiento comercial con el resto de las sucursales.

In [19]:
if tiendas_baja_satisfaccion.empty:
    print("No se encontraron tiendas con satisfacción inferior al 60 %.")
else:
    for _, fila in tiendas_baja_satisfaccion.iterrows():
        print(
            f"Tienda {int(fila['ID_Tienda'])}: "
            f"satisfacción = {fila['Satisfacción_Promedio']} %, "
            f"ingresos = ${fila['Ingresos_Totales']:,.2f}. "
            "Recomendación: mejorar atención al cliente, revisar "
            "disponibilidad de productos y tiempos de servicio."
        )

Tienda 5: satisfacción = 55 %, ingresos = $13,000.00. Recomendación: mejorar atención al cliente, revisar disponibilidad de productos y tiempos de servicio.


## 7. Operaciones con NumPy

Para cumplir expresamente con el requisito del ejercicio, la columna `Total_Ventas` se convierte a un array de NumPy mediante `.to_numpy()`.

Sobre este array se calculan:

- la mediana;
- la desviación estándar.

In [20]:
# Convertir Total_Ventas de Pandas a un array NumPy.
ventas_numpy = ventas["Total_Ventas"].to_numpy()

mediana_ventas_numpy = np.median(ventas_numpy)
desviacion_estandar_numpy = np.std(ventas_numpy)

print("Array NumPy de Total_Ventas:")
print(ventas_numpy)

print("\nMediana de las ventas totales:", mediana_ventas_numpy)
print("Desviación estándar de las ventas:", desviacion_estandar_numpy)

Array NumPy de Total_Ventas:
[ 2000  3000  3000  7500  1000  8000 10500  2500  4000  9000]

Mediana de las ventas totales: 3500.0
Desviación estándar de las ventas: 3189.4356867634124


## 8. Simulación de proyecciones de ventas futuras con NumPy

Se establece una semilla para obtener resultados reproducibles.

Como simulación sencilla, cada venta actual se multiplica por un factor aleatorio con media `1.05` (crecimiento esperado aproximado del 5 %) y desviación estándar `0.10`.

In [21]:
# Generador aleatorio reproducible.
rng = np.random.default_rng(seed=42)

# Factores aleatorios para simular ventas futuras.
factores_proyeccion = rng.normal(
    loc=1.05,
    scale=0.10,
    size=ventas_numpy.size
)

ventas_proyectadas = ventas_numpy * factores_proyeccion

print("Ventas actuales:")
print(ventas_numpy)

print("\nVentas futuras simuladas:")
print(np.round(ventas_proyectadas, 2))

Ventas actuales:
[ 2000  3000  3000  7500  1000  8000 10500  2500  4000  9000]

Ventas futuras simuladas:
[ 2160.94  2838.    3375.14  8580.42   854.9   7358.26 11159.23  2545.94
  4193.28  8682.26]


In [22]:
print("Estadísticas de las ventas futuras simuladas:")
print("Media:", round(np.mean(ventas_proyectadas), 2))
print("Mediana:", round(np.median(ventas_proyectadas), 2))
print(
    "Desviación estándar:",
    round(np.std(ventas_proyectadas), 2)
)
print("Máximo:", round(np.max(ventas_proyectadas), 2))
print("Mínimo:", round(np.min(ventas_proyectadas), 2))

Estadísticas de las ventas futuras simuladas:
Media: 5174.84
Mediana: 3784.21
Desviación estándar: 3298.49
Máximo: 11159.23
Mínimo: 854.9


## 9. Conclusiones

El análisis permitió integrar los datos reales de ventas, inventarios y satisfacción de RetailNow.

- Se calcularon las unidades vendidas por producto y por tienda.
- Se calcularon los ingresos totales mediante `Cantidad_Vendida × Precio_Unitario`.
- Se obtuvo un resumen estadístico de las ventas.
- Se calculó la rotación de inventario por tienda y producto y se aplicó el filtro de nivel crítico inferior al 10 %.
- Se relacionó la satisfacción de cada tienda con sus ingresos y se identificaron las tiendas con satisfacción inferior al 60 %.
- Con NumPy se calculó la mediana y la desviación estándar de `Total_Ventas`.
- Se realizó una simulación reproducible de ventas futuras usando un generador aleatorio con semilla fija.

El archivo de ventas proporcionado no contiene una columna de categoría, por lo que no se realizó el análisis opcional por categoría para evitar inventar información que no existe en los datos.